In [ ]:
"""
3-Qubit *Phase-Flip* repetition code
───────────────────────────────────
"""

# ───────────────────────────────────────────────────────────────────────────
# Qiskit imports
# ───────────────────────────────────────────────────────────────────────────
from qiskit import transpile, QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.result import marginal_counts                      # optional helper
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# ───────────────────────────────────────────────────────────────────────────
# 0.  Backend & registers
# ───────────────────────────────────────────────────────────────────────────
service = QiskitRuntimeService()                     # assumes stored IBM token
num_qubits = 5                                       # need ≥ 5 physical qubits
backend = service.least_busy(operational=True,
                             simulator=False,
                             min_num_qubits=num_qubits)
print("Running on backend:", backend.name)

# logical-data register (3 qubits) and 2 ancilla-measure qubits
qreg_data     = QuantumRegister(3,  name="dataq")
qreg_measure  = QuantumRegister(2,  name="measq")
creg_data     = ClassicalRegister(3, name="data")        # final read-out
creg_syndrome = ClassicalRegister(2, name="syndrome")    # stabiliser bits

state_data     = qreg_data[0]       # convenience handle
ancillas_data  = qreg_data[1:]      # the other two data qubits


# ───────────────────────────────────────────────────────────────────────────
# 1.  Circuit-building helper functions
# ───────────────────────────────────────────────────────────────────────────
def build_qc() -> QuantumCircuit:
    """Fresh circuit with all 5 qubits + both classical registers."""
    return QuantumCircuit(qreg_data, qreg_measure,
                          creg_data, creg_syndrome)


def initialize_qubits(circ: QuantumCircuit) -> QuantumCircuit:
    """Start in |1⟩ for visibility (change freely)."""
    circ.x(state_data)          # |1⟩  (use H/RY… here for arbitrary |ψ⟩)
    circ.barrier(qreg_data)
    return circ


def encode_phase_flip(circ: QuantumCircuit,
                      state, ancillas) -> QuantumCircuit:
    """
    Phase-flip code = bit-flip code sandwiched by H on *all* data qubits:
        1) copy computational basis with CXs
        2) H on each data qubit  ⇒  |000⟩,|111⟩ → |+++⟩,|---⟩
    """
    for a in ancillas:
        circ.cx(state, a)
    # change to X-basis
    circ.h(state)
    for a in ancillas:
        circ.h(a)
    circ.barrier(qreg_data)
    return circ


def measure_syndrome_phase(circ: QuantumCircuit) -> QuantumCircuit:
    """
    Detect **one** Z error.
    Trick: put another H on all data qubits → Z becomes X, then measure
    parity exactly like the bit-flip template.
    """
    circ.h(qreg_data)                # Z-error ⇢ X-error
    circ.barrier(qreg_data, qreg_measure)

    # parity checks (same pattern as your bit-flip code)
    circ.cx(qreg_data[0], qreg_measure[0])
    circ.cx(qreg_data[1], qreg_measure[0])
    circ.cx(qreg_data[0], qreg_measure[1])
    circ.cx(qreg_data[2], qreg_measure[1])

    circ.barrier(qreg_data, qreg_measure)
    circ.measure(qreg_measure, creg_syndrome)

    # fast reset of the measure qubits so we can reuse them (optional)
    with circ.if_test((creg_syndrome[0], 1)):
        circ.x(qreg_measure[0])
    with circ.if_test((creg_syndrome[1], 1)):
        circ.x(qreg_measure[1])

    circ.barrier(qreg_data, qreg_measure)
    return circ


def apply_correction_phase(circ: QuantumCircuit) -> QuantumCircuit:
    """
    Conditional **Z** flips (not X!) because we are correcting phase errors.
    Syndrome bit-mapping:
        11 = error on q0,  01 = q1,  10 = q2
    """
    with circ.if_test((creg_syndrome, 3)):   # 11₂
        circ.z(qreg_data[0])
    with circ.if_test((creg_syndrome, 1)):   # 01₂
        circ.z(qreg_data[1])
    with circ.if_test((creg_syndrome, 2)):   # 10₂
        circ.z(qreg_data[2])
    circ.barrier(qreg_data)
    return circ


def apply_final_readout(circ: QuantumCircuit) -> QuantumCircuit:
    """
    *Optional* decoding step: another H on data qubits brings them back
    to computational basis so we can read |ψ⟩ directly.
    """
    circ.h(qreg_data)                 # undo the last set of H gates
    circ.barrier(qreg_data)
    circ.measure(qreg_data, creg_data)
    return circ


def build_error_correction_sequence(apply_correction: bool = True,
                                    inject_error_idx: int | None = 0
                                    ) -> QuantumCircuit:
    """
    Assembles the whole pipeline.
        * set inject_error_idx = None to run the 'no-fault' reference
    """
    circ = build_qc()
    circ = initialize_qubits(circ)
    circ = encode_phase_flip(circ, state_data, ancillas_data)

    # ▼▼  Deliberately insert a *single* phase-flip for testing  ▼▼
    if inject_error_idx is not None:
        circ.z(qreg_data[inject_error_idx])
    circ.barrier(qreg_data)

    circ = measure_syndrome_phase(circ)

    if apply_correction:
        circ = apply_correction_phase(circ)

    circ = apply_final_readout(circ)
    return circ


# ───────────────────────────────────────────────────────────────────────────
# 2.  Build, draw, and transpile
# ───────────────────────────────────────────────────────────────────────────
circuit = build_error_correction_sequence(apply_correction=True,
                                          inject_error_idx=0)  # Z on q0
print("\nEncoded-→Error-→Detect-→Correct circuit:")
display(circuit.draw("mpl", style="iqp"))

transpiled = transpile(circuit,
                       backend=backend,
                       optimization_level=3)

# ───────────────────────────────────────────────────────────────────────────
# 3.  Run via Runtime SamplerV2 (fast, returns quasi-dists)
# ───────────────────────────────────────────────────────────────────────────
sampler = Sampler(backend)
job     = sampler.run([transpiled], shots=1024)
quasi   = job.result().quasi_dists[0]           # dictionary {bit-string: prob}

# convert to integer-keyed counts for clarity
counts = {int(k, 16): int(v * 1024) for k, v in quasi.items()}
print("\nSampled counts (data qubits + syndrome bits):")
print(counts)

# Example post-processing: keep only the three data-qubit bits
data_counts = marginal_counts(counts, indices=[0, 1, 2])  # little-endian
print("Marginal on data qubits:", data_counts)


IBMNotAuthorizedError: '401 Client Error: Unauthorized for url: https://auth.quantum.ibm.com/api/users/loginWithToken. License required. You need to accept the License., Error code: 3444.'